# Ingesta de personas

Lee `person.json` (JSON) desde Bronze, aplica un esquema explícito, estandariza las columnas y guarda el resultado como Parquet en `silver/persons` (modo `overwrite`).

### Configuración

Carga las rutas de almacenamiento y las funciones comunes, y lee el parámetro `p_environment`.

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
from pyspark.sql.functions import col, lit, concat, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType 

### Lectura del archivo

Lectura inicial de `person.json` con inferencia para inspeccionar su estructura anidada.

In [0]:
person_df = (
    spark.read
        .format("json")
        .load(f"{bronze_folder_path}/person.json")
)
display(person_df)

### Exploración del campo anidado

Prueba la extracción de `forename` y `surname` desde `personName` y su concatenación en `fullname`.

In [0]:
(
    person_df.select(
        col("personId").alias("person_id"),
        col("personName.forename").alias("forename"),
        col("personName.surname").alias("surname")
    )
    .withColumn("fullname",
        concat(col("forename"), lit(" "), col("surname"))
    )
    .show()
)

### Lectura con esquema explícito

Define `personName` como `StructType` anidado y relee el archivo.

In [0]:
personName_shema = StructType([
    StructField("forename", StringType(), True),
    StructField("surname", StringType(), True)
])
person_schema = StructType([
    StructField("personId", IntegerType(), True),
    StructField("personName", personName_shema, True)
])
person_df = (
    spark.read
        .schema(person_schema)
        .format("json")
        .load(f"{bronze_folder_path}/person.json")
)
person_df.printSchema()
display(person_df)

### Transformación de columnas

Une `forename` y `surname` en `fullname`, renombra `personId` y agrega `ingestion_date` y `env`.

In [0]:
person_df = (
    add_ingestion_date(person_df)
        .select(
            col("personId").alias("person_id"),
            concat(col("personName.forename"), lit(" "), col("personName.surname")).alias("fullname"),
            lit(v_env).alias("env")
).drop("personName")
)

display(person_df)

### Escritura en Silver

Guarda el resultado como Parquet en `silver/persons` en modo `overwrite`.

In [0]:
(
    person_df.write
        .format("parquet")
        .mode("overwrite")
        .save(f"{silver_folder_path}/persons")
)


### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")